# 🧑 Face Recognition
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/face-recognition/training/notebook.ipynb)

Who is in this photo, and do two photos show the same person? Every network of the pipeline is
trained in this notebook:
1. a **face detector** finds each face and its five landmarks (eyes, nose tip, mouth corners);
2. the face is rotated and scaled onto a fixed template;
3. an ensemble of two **face embedders** (an EfficientNet-B2 that starts from ImageNet weights and a
   ResNet-18 trained from scratch, both with a CosFace margin loss) turns it into an embedding;
4. a small **MLP head** names one of the 42 people of LFW (Labeled Faces in the Wild) with at least
   25 photos; verification compares two embeddings with the cosine distance.

No pretrained face network is part of the model. The 42 people's test photos and a set of
never-seen people are held out from everything, including the detector.

This notebook is the controller: every step calls the reusable code in `src/` and the model code
in `../model/model.py` (the same file the Space and the Inference Endpoint run).

**Contents:** 1 Setup · 2 Config · 3 Data · 4 Load model · 5 Training · 6 Evaluation · 7 Inference · 8 Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "face-recognition"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
# 1d. Project code: src/ (training) + ../model/model.py (imported as `model`, the deployed code)
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

from src import data_setup, detector, engine, export, model_builder, utils
from src.config import Config
import model as M

print({**utils.lib_versions("torch", "torchvision", "sklearn", "numpy", "huggingface_hub"), "device": utils.device_name("torch")})

## 2. Config
Every hyperparameter lives in `src/config.py`. `SMOKE_TEST=1` (environment or the cell below) runs
the same pipeline on a tiny subset (60 Open Images photos, 6 known + 60 other LFW people, 1-2
epochs) and exports to `outputs/smoke/model`, never to `../model`. `PUSH_TO_HUB = True` uploads the
exported folder at the end (full runs only). The full run takes about 4 hours on a GPU (detector
~2 h, embedders ~1.8 h); on a CPU it is far longer.

In [ ]:
import os
os.environ.setdefault("SMOKE_TEST", "0")   # "1" = few-minute sanity run
PUSH_TO_HUB = False

cfg = Config()
utils.set_seeds(cfg.seed, "torch")
print("export to:", cfg.model_dir.relative_to(utils.PROJECT_DIR))
pd.Series(cfg.to_dict()).to_frame("value")

## 3. Data
- **LFW**: 13,233 photos of 5,749 people (downloaded once, ~233 MB, into `training/data/sklearn_lfw`).
  The splits are made first, so the recognition test photos and the unseen people are kept out of
  every training step below.
- **Open Images** (Google; annotations CC BY 4.0, images CC BY 2.0): the ~12,000 validation + test
  photos with a human-drawn "Human face" box (downloaded once, ~1.2 GB, into `training/data/openimages`).
  They teach the detector faces of every size, pose and crowd.
- **Landmark labels** (`training/labels/landmark_labels.json.gz`, committed): the 5 points of the
  faces in these photos, made once with a reference detector (MTCNN, facenet-pytorch). Our detector
  learns from these numbers and from the human boxes; no third-party code or weights are used.

In [ ]:
paths, names = data_setup.list_photos(cfg)
splits = data_setup.make_splits(cfg, names)
class_names = splits["class_names"]
y = {k: data_setup.labels(names, splits[k], class_names) for k in ("train", "val", "test")}
print(f"LFW: {len(names):,} photos of {len(set(names)):,} people | {len(class_names)} people to name")
pd.Series(data_setup.split_summary(splits, names)).to_frame("split")

In [ ]:
oi_ann = detector.download_open_images(cfg)
oi_labels, lfw_labels = detector.landmark_labels(cfg)
lfw_photos = data_setup.lfw_detector_photos(paths, splits)
records = detector.build_records(cfg, oi_ann, oi_labels, lfw_photos, lfw_labels)
det_train, det_val = detector.split_records(cfg, records)
print(f"detector data: {len(det_train):,} training photos ({sum(len(r[1]) for r in det_train):,} faces), "
      f"{len(det_val)} held-out Open Images photos")

In [ ]:
# A few training photos: human boxes (green), ignored faces (orange), landmark labels (red)
import matplotlib.pyplot as plt
from PIL import Image as PILImage
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, rec in zip(axes, [r for r in det_train if cfg.open_images_dir in r[0].parents][:4]):
    ax.imshow(PILImage.open(rec[0])); ax.axis("off")
    for b, l, h, ig in zip(*rec[1:]):
        ax.add_patch(plt.Rectangle(b[:2], b[2] - b[0], b[3] - b[1], fill=False, lw=1.5, color="orange" if ig else "lime"))
        if h:
            ax.scatter(l[0::2], l[1::2], s=6, c="red")
plt.show()

## 4. Load model
- `model.FaceDetector`: a single-shot CNN (feature pyramid at strides 8/16/32, anchors) that
  predicts, for every anchor, face / not face, a box and 5 landmarks. Random initial weights.
- `model.FaceRecognizer`: one member per `cfg.embedders` entry (a torchvision backbone, ImageNet
  weights when `pretrained`, + a Linear/BatchNorm projection to 512-d) and the identity head on the
  concatenated embeddings (1024 -> 256 -> 42).

In [ ]:
det = model_builder.build_detector(device)
net = model_builder.build_recognizer(cfg, class_names).to(device).to(memory_format=torch.channels_last)
params = model_builder.pipeline_params(det, net)
print("embedders:", ", ".join(model_builder.member_name(e) for e in cfg.embedders))
pd.Series(params).map("{:,}".format).to_frame("parameters")

## 5. Training
**Stage 0, face detector.** Random crops, zoom-outs and 2x2 mosaics at 640 px. Per anchor: face /
not face with online hard-negative mining (the 3 hardest negatives per face count), box and
landmark regression. Every 3 epochs it is scored on the held-out Open Images photos; the best
epoch is kept. Then every LFW photo is aligned with it (cached).

**Stage 1, embedders**, one after the other. CosFace (a softmax with a margin on the cosine to
each identity's centre) over every training identity, with random zoom/shift/rotation/flip/colour/
erasing on the device. After every epoch the embeddings are checked on the 42 people (nearest-centre
accuracy, train vs validation) and on the unseen people (verification AUC); the best epoch on
validation is kept.

**Stage 2, identity head.** The MLP learns the 42 people from the frozen ensemble embeddings; the
epoch with the lowest validation loss is kept. The verification threshold is also picked on
validation pairs. The test split is not used until section 6.

In [ ]:
import time
start = time.perf_counter()
det_history = detector.train_detector(det, det_train, det_val, cfg, device)
print(f"detector training: {(time.perf_counter() - start) / 60:.1f} min")
plots = utils.OUTPUTS_DIR / ("smoke/plots" if cfg.smoke else "plots")
display(Image(str(export.plot_detector_history(det_history, plots / "detector_training.png")), width=900))

In [ ]:
# Align every LFW photo with the detector we just trained (the most central large face; the Space
# uses the largest face). Cached, keyed to the detector weights.
aligner = model_builder.build_aligner(cfg, det)
data = data_setup.align_photos(cfg, aligner, model_builder.detector_id(det))
crops_np = data["crops"]
assert (data["names"] == names).all()
print(f"no face found in {int((~data['found']).sum())} of {len(names):,} photos (center-crop fallback)")
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, i in zip(axes.ravel(), np.random.default_rng(0).choice(len(names), 16, replace=False)):
    ax.imshow(crops_np[i]); ax.set_title(names[i], fontsize=6); ax.axis("off")
plt.show()
crops = torch.from_numpy(crops_np).permute(0, 3, 1, 2).to(device)    # uint8 (N, 3, 128, 128), on the device

In [ ]:
histories = {}
for i, (member, spec) in enumerate(zip(net.members, cfg.embedders), 1):
    name = f"{i}. {model_builder.member_name(spec)}"
    print(f"--- {name}")
    histories[name], _ = engine.train_embedder(member, spec, crops, names, splits, cfg, device)
    print(f"{(time.perf_counter() - start) / 60:.1f} min so far")

In [ ]:
emb = {k: engine.embed(net, crops, splits[k], device) for k in ("train", "val", "test", "unseen")}
head_history = engine.train_head(net, emb, y, cfg, device)
train_time_s = time.perf_counter() - start

net.set_verify_threshold(engine.tune_verify_threshold(emb["val"], y["val"], cfg.verify_grid))
print("verification threshold (cosine distance, picked on validation pairs):", net.verify_threshold)
display(Image(str(export.plot_history(histories, head_history, plots / "training_curves.png")), width=900))
utils.save_json({"detector": det_history, "embedders": histories, "head": head_history}, plots.parent / "histories.json")

## 6. Evaluation
**Detection:** AP at IoU 0.5 for large (>= 64 px), medium (>= 32 px) and small (>= 16 px) faces, on
the held-out Open Images photos and on WIDER FACE val (3,226 photos; evaluation only, CC BY-NC-ND,
downloaded once into `training/data/wider_face`), next to the reference detector's scores on the
same photos (`config.REFERENCE_DETECTOR_AP`).

**Identification:** top-1 accuracy, macro F1 and top-3 accuracy on train / validation / test (the gap
between them shows overfitting). **Verification:** every pair of test faces, and every pair of faces
of the **unseen** people, is a match when its cosine distance is below the threshold.

In [ ]:
from src.config import REFERENCE_DETECTOR_AP
det_scores = {"held-out Open Images": {
    "ours": detector.evaluate_detections(detector.detector_fn(det), detector.held_out_items(det_val)),
    "reference (MTCNN)": REFERENCE_DETECTOR_AP["held-out Open Images"]}}
if cfg.eval_wider:
    items = detector.wider_items(cfg, cfg.wider_images)
    det_scores["WIDER FACE val"] = {"ours": detector.evaluate_detections(detector.detector_fn(det), items),
                                    "reference (MTCNN)": REFERENCE_DETECTOR_AP["WIDER FACE val"]}
    display(Image(str(export.plot_detector_ap(det_scores["WIDER FACE val"], plots / "detector_ap.png")), width=720))
pd.DataFrame({(s, d): v for s, per in det_scores.items() for d, v in per.items()}).T.round(4)

In [ ]:
prob = {k: engine.head_proba(net, emb[k], device) for k in ("train", "val", "test")}
scores = {k: engine.classification_metrics(y[k], prob[k]) for k in prob}
verification = engine.verification_metrics(emb["test"], y["test"], net.verify_threshold)
verification_unseen = engine.verification_metrics(emb["unseen"], names[splits["unseen"]], net.verify_threshold)
display(pd.DataFrame(scores).T.round(4))
pd.DataFrame({"test (42 people)": verification, "unseen people": verification_unseen})

In [ ]:
# Every embedder that was tried, all with this code and these splits (see src/config.EXPERIMENTS)
from src.config import EXPERIMENTS
display(pd.DataFrame(EXPERIMENTS).T)
pred = prob["test"].argmax(1)
wrong = np.flatnonzero(pred != y["test"])
print("test mistakes:", [(class_names[y["test"][i]], "->", class_names[pred[i]]) for i in wrong])
labels_all = data_setup.labels(names, np.concatenate([splits["train"], splits["val"], splits["test"]]), class_names)
for path in filter(None, (export.plot_experiments(EXPERIMENTS, plots / "model_comparison.png"),
             export.plot_confusion_matrix(y["test"], pred, class_names, plots / "confusion_matrix.png"),
             export.plot_verification(emb["test"], y["test"], net.verify_threshold, plots / "verification_distances.png"),
             export.plot_tsne(np.concatenate([emb["train"], emb["val"], emb["test"]]), labels_all,
                              plots / "tsne_embeddings.png", cfg.tsne_max_points, cfg.seed))):
    display(Image(str(path), width=720))

## 7. Inference
Save the detector and the recognizer into `cfg.model_dir` and load them exactly as the Space and
the Inference Endpoint do: `model.load(dir, device)`, then `predict` / `verify` / `embed` on the
photos in `../space/examples`.

In [ ]:
det, net = det.cpu().eval(), net.cpu().to(memory_format=torch.contiguous_format).eval()
export.save_model(det, net, cfg.model_dir)
predictor = M.load(cfg.model_dir, device)

examples = sorted((utils.SPACE_DIR / "examples").glob("*.jpg"))
for path in examples:
    top3 = dict(list(predictor.predict(path).items())[:3])
    print(f"{path.name:22s} -> " + ", ".join(f"{k}: {v:.3f}" for k, v in top3.items()))
print("verify(same photo twice):", predictor.verify(examples[0], examples[0]))
print("verify(two people):      ", predictor.verify(examples[0], examples[1]))
print("embed ->", len(predictor.embed(examples[0])), "floats")

# The loaded predictor reproduces the in-memory model on the test embeddings:
print("same test predictions as the trained head:", bool((predictor.predict_embeddings(emb["test"]).argmax(1) == pred).all()))

## 8. Export
Writes `face_detector.safetensors`, `model.safetensors` + `config.json`, `metrics.json` (deployed
model, detector scores and every experiment) and the graphs in `assets/`, and refreshes the metrics
in the model card. `PUSH_TO_HUB = True` then uploads the folder to `shalev396/face-recognition`
(token from `hf auth login` or an `HF_TOKEN` secret/env var).

In [ ]:
today = utils.today()
source = f"smoke run {today}" if cfg.smoke else f"trained {today} with training/ (local {utils.device_name('torch').upper()})"
metrics = export.build_metrics(
    cfg, {**scores["test"], **{k: v for k, v in verification.items() if not k.startswith("n_")},
          "unseen_verify_roc_auc": verification_unseen["verify_roc_auc"]},
    data={"n_photos": len(names), "n_people": len(set(names)), "n_classes": len(class_names),
          **{f"n_{k}": len(splits[k]) for k in ("embedder_train", "train", "val", "test", "unseen")},
          "embedder_identities": len(splits["embedder_identities"]), "no_face_found": int((~data["found"]).sum()),
          "detector_train_photos": len(det_train), "detector_train_faces": int(sum(len(r[1]) for r in det_train)),
          "detector_held_out_photos": len(det_val)},
    params=params, train_time_s=train_time_s, device=utils.device_name("torch"), source=source,
    experiments=EXPERIMENTS,
    extra={"splits": {k: scores[k] for k in ("train", "val")},
           "detector": {"best_epoch": det_history["best_epoch"], "ap_iou_0.5": det_scores},
           "embedders": {n: {**spec, "best_epoch": h["best_epoch"]} for (n, h), spec in zip(histories.items(), cfg.embedders)},
           "head_best_epoch": head_history["best_epoch"],
           "test_mistakes": [[class_names[y["test"][i]], class_names[pred[i]]] for i in wrong],
           "verification": {"threshold": net.verify_threshold, "chosen_on": "validation pairs (best balanced accuracy)",
                            "test": verification, "unseen_people": verification_unseen}})
export.export(det, net, cfg.model_dir, metrics, class_names=class_names, y_true=y["test"], y_pred=pred,
              history=histories, head_history=head_history, detector_history=det_history,
              detector_ap=det_scores.get("WIDER FACE val"), experiments=EXPERIMENTS,
              tsne_embeddings=np.concatenate([emb["train"], emb["val"], emb["test"]]), tsne_labels=labels_all,
              test_embeddings=emb["test"], test_labels=y["test"], verify_threshold=net.verify_threshold,
              tsne_max_points=cfg.tsne_max_points, seed=cfg.seed)
sorted(p.relative_to(cfg.model_dir).as_posix() for p in cfg.model_dir.rglob("*")
       if p.is_file() and "__pycache__" not in p.parts and ".git" not in p.parts)

In [ ]:
if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))
else:
    print("not uploading (PUSH_TO_HUB is False or this is a smoke run)")